# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and analyzing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset schema is provided via the Croissant JSON-LD URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```

**Note:** All entities (record sets, fields, columns, etc.) in this exploration are referenced by their `@id`, as recommended for FAIR datasets.

In [ ]:
# If `mlcroissant` is not installed, install it now
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object. Print summary details.
metadata = dataset.metadata
print('Dataset Name:', getattr(metadata, 'name', '[no name]'))
print('Description:', getattr(metadata, 'description', '[no description]'))
print('Identifier:', getattr(metadata, 'identifier', '[no identifier]'))
print('Authors (@id):', getattr(metadata, 'author', '[no authors]'))
print('RecordSets (@id):', getattr(metadata, 'recordSet', '[no record sets]'))

# For deeper inspection
# print(json.dumps(metadata.to_json(), indent=2))

## 2. Data Overview
Review available record sets, their fields, and associated entity `@id`s.

@id referencing is used for all entities for clarity.

In [ ]:
# List all record set @ids
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    print('No record sets found.')
else:
    print('Record sets available:')
    for rs in record_sets:
        print('- @id:', rs['@id'])

    # For each record set, list fields and columns by @id
    for rs in record_sets:
        print('\nRecordSet:', rs['@id'])
        fields = rs.get('field', [])
        if fields:
            print('Fields:')
            for field in fields:
                print('  - @id:', field['@id'], '| name:', field.get('name', '[no name]'))

                columns = field.get('column', [])
                if columns:
                    print('    Columns:')
                    for col in columns:
                        print('      - @id:', col['@id'], '| name:', col.get('name', '[no name]'))
        else:
            print('  No fields found for this record set.')

## 3. Data Extraction
Load each record set into a DataFrame for analysis. Use the record set and field `@id`s established above.

For demonstration, we will extract and show all available record sets.

In [ ]:
# Extract data from each record set
dfs = {}

record_sets = getattr(metadata, 'recordSet', [])
record_set_ids = [rs['@id'] for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dfs[record_set_id] = pd.DataFrame(records)

# Print column names from each record set
for rs_id, df in dfs.items():
    print(f"\nRecordSet @id: {rs_id}\nColumns: {df.columns.tolist()}")
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process data from a record set. We'll demonstrate filtering, normalization, and grouping operations.

**Note:** Adjust the `record_set_id`, `numeric_field`, and `group_field` based on the actual schema. Here we select the first record set for example purposes.

In [ ]:
# Select a record set to analyze
if record_set_ids:
    rs_id = record_set_ids[0]
    df = dfs[rs_id]
    print(f"Exploring record set: {rs_id}")
    print(f"Available columns: {df.columns.tolist()}")

    # Choose a numeric field (by column @id or column name, here assumed 'age_at_diagnosis' as example)
    numeric_field = None
    numeric_candidates = ['age_at_diagnosis', 'height', 'hormone_level']  # Example guesses
    for col in df.columns:
        if any(candidate in col for candidate in numeric_candidates):
            numeric_field = col
            break

    if numeric_field is None:
        print('No numeric field detected for EDA demo.')
    else:
        threshold = 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Attempt grouping by a categorical field (e.g. 'clitomegaly', 'infertility', or similar)
        group_field = None
        group_candidates = ['clitomegaly', 'hirsutism', 'infertility', 'pathogenicity_assessment']
        for col in df.columns:
            if any(candidate in col for candidate in group_candidates):
                group_field = col
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
        else:
            print('No suitable group field found.')
else:
    print('No record sets found for EDA.')

## 5. Visualization
Visualize the distribution of the normalized field and relationships between selected attributes.

Example: Histogram of normalized age at diagnosis, scatter plot comparing normalized numeric fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize for the selected record set and fields
if record_set_ids and numeric_field:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[norm_col], bins=10, kde=True)
    plt.title(f'Histogram of {norm_col} in {rs_id}')
    plt.xlabel(norm_col)
    plt.ylabel('Frequency')
    plt.show()

    # Try scatter plot between two numeric fields
    other_numeric = None
    for col in df.columns:
        if col != numeric_field and pd.api.types.is_numeric_dtype(df[col]):
            other_numeric = col
            break
    if other_numeric:
        plt.figure(figsize=(6,4))
        sns.scatterplot(data=filtered_df, x=numeric_field, y=other_numeric, hue=group_field if group_field else None)
        plt.title(f'Scatter plot of {numeric_field} vs {other_numeric}')
        plt.xlabel(numeric_field)
        plt.ylabel(other_numeric)
        plt.show()
else:
    print('Cannot visualize: no suitable numeric field or record set.')

## 6. Conclusion
- Loaded FAIR^2 clinical dataset using `mlcroissant` from Croissant schema.
- Analyzed data structure: record sets, fields, columns identified by `@id`.
- Demonstrated basic data extraction, processing and normalization of numeric fields.
- Provided visualizations of selected attributes.

For more advanced use, refer to the Croissant schema for precise field and column `@id` values, explore further processing based on rich metadata, or integrate with clinical/genomic analysis tools.